In [ ]:
# 💡 데이터셋 이름: idiotDeveloper/Korean_Telephone_Data_10000
# 🎯 데이터셋 의미: 방대한 양의 한국어 전화 통화 녹음(Audio)과 그에 대한 스크립트(Transcripts)가 쌍으로 매칭된 데이터셋입니다.
# 📝 학습 목표: 이 데이터셋을 활용하여, 통화 스크립트의 길이 분포를 분석하고, 데이터 처리 과정을 실습하는 방법을 배웁니다.
# 🤖 튜터의 코멘트: 이 데이터는 음성 인식(Speech Recognition) 및 자연어 처리(NLP) 분야의 기초 자료입니다. 우리는 오늘 '텍스트 분석'에 초점을 맞출 거예요!

# 필요한 라이브러리 불러오기
from datasets import load_dataset, DatasetDict
import random
import numpy as np
import time

# --- 설정 변수 ---
DATASET_NAME = "idiotDeveloper/Korean_Telephone_Data_10000"
SAMPLE_COUNT = 100  # 분석을 위해 상위 100개의 샘플만 사용합니다.
TARGET_SPLIT = 'test' # 스트리밍 테스트를 위해 'test' 스플릿을 우선 사용합니다.

def load_data_safely(dataset_name, target_split):
    """
    데이터 로딩을 시도하며, 스트리밍 실패 시 일반 모드로 대체하는 안전 장치입니다.
    """
    print("🚀 데이터셋 로딩을 시작합니다. (스트리밍 우선 시도...)")
    
    # 1. 스트리밍 모드로 로딩 시도 (가장 빠르고 메모리 효율적)
    try:
        dataset = load_dataset(dataset_name, split=target_split, streaming=True)
        print("✅ [SUCCESS] 스트리밍 모드(streaming=True)로 성공적으로 연결되었습니다! (메모리 절약 최고!)")
        return dataset
    except Exception as e:
        # 2. 스트리밍 실패 시 일반 로딩으로 대체 (소량만 메모리에 로드)
        print(f"⚠️ [WARNING] 스트리밍 로드 실패 ({type(e).__name__} 발생). 일반 로딩 방식으로 대체합니다...")
        try:
            # 전체 데이터셋을 다 받기 어려우므로, 'test' 스플릿만 비스트리밍으로 로드합니다.
            dataset = load_dataset(dataset_name, split=target_split)
            print(f"✅ [SUCCESS] {target_split} 스플릿을 일반 모드(Dataset)로 로드했습니다. (작은 분량만 로드!)")
            return dataset
        except Exception as e_fallback:
            print(f"❌ [ERROR] 모든 로딩 방식에서 실패했습니다: {e_fallback}")
            return None

def run_analysis(dataset_iterator, sample_count):
    """
    주어진 데이터셋 이터레이터에서 핵심 통계 분석을 수행합니다.
    """
    print("\n=============================================================================")
    print(f"🧠 [STEP 2] 데이터 샘플링 및 정량적 분석 ({sample_count}개 샘플 분석 시작)")
    print("=============================================================================")

    # ⭐️ 필수 구조 패턴 적용: take()가 있는지 확인하여 스트리밍/일반 모드에 맞는 샘플링을 합니다.
    if hasattr(dataset_iterator, "take"):
        print("✨ 패턴 적용: take() 함수가 존재합니다. (스트리밍 데이터셋으로 추정)")
        sampled_dataset_iterator = dataset_iterator.take(sample_count)
    else:
        print("✨ 패턴 적용: take() 함수가 존재하지 않습니다. (일반 Dataset으로 추정)")
        sampled_dataset_iterator = dataset_iterator.take(sample_count)
        
    # --- 데이터 분석 변수 초기화 ---
    word_counts = []
    transcript_lengths = []
    audio_durations = []
    
    processed_count = 0
    
    # 💡 샘플 데이터를 메모리에서 직접 뽑아오는 과정을 시뮬레이션합니다.
    print("\n🔍 상위 샘플들을 순차적으로 탐색하며 분석합니다...")
    
    # 🔄 이터레이터를 사용하여 상위 K개의 샘플만 처리합니다.
    sample_data_list = []
    for i, sample in enumerate(sampled_dataset_iterator):
        if i >= sample_count:
            break
        
        sample_data_list.append(sample)
        
        # 1. 스크립트 텍스트 분석 (주요 분석 대상)
        transcript = sample['transcripts']
        word_count = len(transcript.split())
        word_counts.append(word_count)
        
        # 2. 길이 측정 (문자열 분석)
        transcript_lengths.append(len(transcript))
        
        # 3. 오디오 길이 분석 (시간 정보 추출)
        # audio 필드 구조: {'sampling_rate': 16000, 'array': numpy_array}
        if 'audio' in sample and 'array' in sample['audio']:
             # numpy 배열의 크기 (샘플링된 시간)를 길이로 간주합니다.
            sample_audio_length = len(sample['audio']['array']) / sample['audio']['sampling_rate']
            audio_durations.append(sample_audio_length)


    # --- 📈 정량적 분석 결과 출력 ---
    print("\n=============================================================================")
    print("🌟 분석 결과: 통계 요약 (Stats Summary)")
    print("=============================================================================")
    
    print(f"➡️ 총 분석 샘플 개수: {len(sample_data_list)}개")
    
    # 1. 단어 수 분석
    avg_words = np.mean(word_counts)
    print(f"📊 1. 평균 단어 수: {avg_words:.2f} 단어")
    print(f"   (⭐ 이 데이터셋의 스크립트는 평균적으로 {avg_words:.2f} 단어 길이입니다.)")
    
    # 2. 문자열 길이 분석
    avg_chars = np.mean(transcript_lengths)
    print(f"📊 2. 평균 문자 길이: {avg_chars:.2f} 문자")
    
    # 3. 오디오 길이 분석
    avg_audio_time = np.mean(audio_durations)
    print(f"📊 3. 평균 오디오 시간: {avg_audio_time:.2f} 초")
    
    # 🧠 창의적 해석: 단어 수와 오디오 시간의 관계 추론
    print("\n🧠 [TUTOR INSIGHT] 데이터 속 숨겨진 비밀 찾기:")
    print("   👉 이 결과들을 보면, 텍스트의 길이(단어 수)와 실제 음성 시간(오디오 시간)이 어느 정도 비례하는지 알 수 있습니다.")
    print("   👉 만약 평균 오디오 시간이 평균 단어 수보다 현저히 길다면, 침묵(Silence) 시간이 길다는 것을 의미할 수 있어요!")
    
    
    # --- 💬 LLM 프롬프트 생성 예시 (활용 예시) ---
    print("\n=============================================================================")
    print("💡 [STEP 3] AI 응용 실습: 프롬프트 엔지니어링 맛보기")
    print("=============================================================================")
    
    sample_text = sample_data_list[0]['transcripts']
    
    print(f"\n[🔍 분석 샘플 텍스트 (첫 번째): {sample_text[:50]}...]")
    print("-----------------------------------------------------------------------------")
    print("📜 목표: 이 스크립트 내용을 요약하거나, 특정 정보를 추출하는 AI 모델에게 지시문을 작성해봅시다.")
    
    system_prompt = (
        "당신은 한국어 전문 챗봇입니다. 주어진 대화 스크립트를 분석하여 다음의 3가지 정보만 JSON 형식으로 뽑아주세요:\n"
        "1. 주요 주제 (Topic):\n"
        "2. 감정 상태 (Sentiment): (긍정/부정/중립)\n"
        "3. 요청 사항 (Action Item):\n"
    )
    
    user_prompt = f"대화 스크립트: \"{sample_text}\""
    
    print("\n✅ 생성된 Prompt 예시:")
    print("-" * 50)
    print("SYSTEM PROMPT:", system_prompt)
    print("USER PROMPT:", user_prompt)
    print("-----------------------------------------------------------------------------")
    print("🎉 축하합니다! 이렇게 데이터를 분석하고, 그 분석 결과를 바탕으로 AI에게 질문(Prompt)을 설계하는 것이 데이터 과학의 핵심 과정입니다!")
    print("=============================================================================")


# ===========================================================================
# 🚀 메인 실행 로직
# ===========================================================================

# 1. 데이터 로드 (스트리밍/일반 모드 판별)
dataset = load_data_safely(DATASET_NAME, TARGET_SPLIT)

if dataset:
    # 2. 분석 실행
    run_analysis(dataset, SAMPLE_COUNT)
else:
    print("\n😢 분석을 진행할 데이터를 로드하지 못했습니다. 스크립트를 종료합니다.")